In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
WORK='/content/sde-seg'
DRIVE='/content/drive/MyDrive/sde-seg-run'
os.makedirs(DRIVE,exist_ok=True)
%cd /content


In [ ]:
# Upload the prepared sde-seg.zip if the project is not already present.
from pathlib import Path
if not os.path.isdir(WORK):
    from google.colab import files
    uploaded = files.upload()
    zip_names = [n for n in uploaded if n.lower().endswith('.zip')]
    if not zip_names:
        raise FileNotFoundError('Upload sde-seg.zip, then rerun this cell.')
    !unzip -q -o "{zip_names[0]}" -d /content
    # Accept either a top-level sde-seg directory or a zip extracted into the current folder.
    if not os.path.isdir(WORK):
        candidates = [p for p in Path('/content').iterdir() if p.is_dir() and (p/'requirements.txt').exists()]
        if candidates:
            WORK = str(candidates[0])
    if not os.path.isdir(WORK):
        raise FileNotFoundError('Could not find the extracted project root containing requirements.txt.')
%cd /content/sde-seg
!pip install -r requirements.txt
!pip install huggingface_hub

In [ ]:
# Persist outputs and checkpoints on Drive.
!mkdir -p data/raw results /content/drive/MyDrive/sde-seg-run/{checkpoints,logs}
!rm -rf results/checkpoints results/logs
!ln -s /content/drive/MyDrive/sde-seg-run/checkpoints results/checkpoints
!ln -s /content/drive/MyDrive/sde-seg-run/logs results/logs
import os, sys
os.environ['PYTHONPATH'] = os.getcwd() + ':' + os.environ.get('PYTHONPATH', '')
!{sys.executable} -m pytest -q

In [ ]:
# In-The-Wild
!mkdir -p data/raw/in_the_wild
!wget -O /content/itw.zip 'https://huggingface.co/datasets/mueller91/In-The-Wild/resolve/main/release_in_the_wild.zip'
!unzip -q -o /content/itw.zip -d data/raw/in_the_wild

# ML-ITW
!mkdir -p data/raw/ml_itw
!huggingface-cli download chlo1/ML-ITW --repo-type dataset --local-dir data/raw/ml_itw


In [ ]:
# ASVspoof LA: paste the official direct LA.zip URL below.
LA_URL = ''
if not LA_URL: print('Paste the official LA.zip direct URL into LA_URL, then rerun this cell.')
else:
  !wget -O /content/LA.zip "{LA_URL}"
  !unzip -q -o /content/LA.zip -d data/raw/LA


In [ ]:
!python scripts/check_data.py
import pandas as pd, pathlib
for p in pathlib.Path('data/raw').rglob('meta.csv'):
    d=pd.read_csv(p); print(p, list(d.columns)); [print(c, d[c].dropna().astype(str).unique()[:30]) for c in d.columns]


In [ ]:
# Smoke test: one epoch, 500 training utterances. Checkpoint is written every epoch.
!python -m sdeseg.train --data-root data/raw/LA --variant full --seed 0 --smoke --epochs 1 --checkpoint-dir results/checkpoints


In [ ]:
# Full resumable runs. Re-running a cell resumes from *_latest.pt automatically.
for variant in ['full', 'no_segm', 'segm_no_stab']:
  for seed in [0, 1, 2]:
    !python -m sdeseg.train --data-root data/raw/LA --variant {variant} --seed {seed} --checkpoint-dir results/checkpoints


In [ ]:
# Package generated outputs for download.
!mkdir -p /content/sde-seg-download
!zip -qr /content/sde-seg-results.zip results/tables results/figures results/logs
from google.colab import files
files.download('/content/sde-seg-results.zip')
